In [2]:
!git config --global user.name "Akintoye Felix"
!git config --global user.email "akintoyesylvester1996@gmail.com"

In [4]:
!git init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/


In [5]:
!git status

On branch master

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.config/
	Github_Repos_Access_Token.txt
	drive/
	sample_data/

nothing added to commit but untracked files present (use "git add" to track)


In [ ]:
!git

multi-agent application using crewAI
using 80/20 rules : Task first , Agent second
Role -- Goal -- BackStory -- Framework
Start Sequential; add hierarchical only when needed

Installation of dependencies

In [ ]:
%pip install --quiet crewai crewai-tools elevenlabs python-dotenv pydub pydantic
%pip install ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.8/413.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 701.3/701.3 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 628.3/628.3 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 858.7/858.7 kB 38.6 MB/s 

In [ ]:
# %pip install requests==2.32.4
# %pip install "tokenizers>=0.21,<0.22"

**APIs needed:**

1.Elevenlabs key  -- voices augment

2.Serper dev key -- web search

3.OpenAI key  

In [ ]:
import os
from google.colab import userdata

# setting the api keys as environment variables
os.environ['Elevenlabs_API_keys'] = userdata.get('Elevenlabs_API_keys')
os.environ['Serper_API_key'] = userdata.get('Serper_API_key')
os.environ['OpenAI_API_key'] = userdata.get('OpenAI_API_key')


# voice ids from elevenlabs

os.environ['Lisa_M_Voice_ID']=userdata.get('Lisa_M_Voice_ID')
os.environ['Mark_Voice_ID']=userdata.get('Mark_Voice_ID')

importing required tools/libraries

In [ ]:
from typing import List, Dict, Optional, Any, Type
from datetime import datetime
from pydub import AudioSegment  #To edit the audios
from crewai.tools import BaseTool # basetool allow crewai to create tool that can be used directly
from pydantic import Field, BaseModel, ConfigDict #used to create pydantic model
from elevenlabs.client import ElevenLabs

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [ ]:

class voiceConfig(BaseModel):
  """ creating a voice configuration settings """

  stability: float =0.46 #slightly lower for more natural variation
  similarity_boost: float = 0.85 #Higher to maintain consistence voice character
  style: float = 0.65 #Balance expression
  use_speaker_boost: bool =True
  model_id: str = "eleven_multilingual_v2"
  ouput_format: str = "mp3_4400_128"
  apply_text_normalization: str ='auto'


class AudioConfig(BaseModel):
  """ audio processing configuration """

  format: str ='mp3'
  sample_rate: int = 48000 # Higher for better quality
  channels: int = 2
  bitrate: str ='256k' # Higher bitrates clearer audio
  target_loudness: float = -14.0 #standard podcast loudness
  compression_rate: float = 2.0  #light compression for voice


class Dialogue(BaseModel):
  """Dialogues for the audio generation tool"""

  speaker: str
  text: str


class PodcastAudioGeneratorInput(BaseTool):
  """Input for the podcast audio generation tool """

  dialogue: list[Dialogue]

In [ ]:
# import typing
# from typing import ClassVar

In [ ]:

class PodcastAudioGeneration(BaseModel):
  """Enhanced podcast audio generation tool."""

  Name: str = "PodcastAudioGenerator"
  Description: str = "Synthetic Voice Genration"

  model_config = ConfigDict(arbitrary_types_allowed=True)

  api_key: str = Field(default_factory=lambda: os.environ['Elevenlabs_API_keys'])

  voice_config: Dict[str, Dict]= Field(default_factory=dict)
  audio_config: AudioConfig = Field(default_factory=AudioConfig) # Use AudioConfig() as default

  output_dir: str = Field(default="output/audio-files")
  client: Any =Field(default=None)

  arg_schema: Type[BaseModel] = PodcastAudioGeneratorInput


  def __init__(self, **data):
    super().__init__(**data)

    if not self.api_key:
      raise ValueError("Elevenlabs API key  variable not instantiated")

    self.client = ElevenLabs(api_key=self.api_key)


  def add_voice(self,name: str, voice_id: str, config: Optional[voiceConfig] = None) -> None:
    """ Add voice to the voice configuration """

    self.voice_config[name] = {
        "voice" : voice_id,
        "config" : config or voiceConfig() # Instantiate voiceConfig
    }

  def run(self, dialogue: List[Dialogue]) -> List[str]: # Use List[Dialogue] for type hinting and return List[str]
    """Genretaes audio  files for each speech segment"""

    os.makedirs(self.output_dir, exist_ok=True) # Use os.makedirs

    audio_files=[]
    for index, segment in enumerate(dialogue):
      speaker=segment.get("speaker", '').strip()
      text=segment.get("text", '').strip()

      if not speaker or not text:
        print(f"skipping segment {index} missing speaker or text")
        continue

      if speaker not in self.voice_config:
        print(f"Skipping segment {index}: No voice configured for speaker '{speaker}'")
        continue


      try:

        audio_generator=self.client.text_to_speech.convert(
            text=text,
            voice_id=self.voice_config[speaker]['voice'], # Correctly access voice_id
            model_id=self.voice_config[speaker]['config'].model_id, # Correctly access model_id
            output_format=self.voice_config[speaker]['config'].ouput_format, # Correctly access output_format

            voice_settings={
                "stability": self.voice_config[speaker]['config'].stability, # Correctly access stability
                "similarity_boost": self.voice_config[speaker]['config'].similarity_boost, # Correctly access similarity_boost
                "style": self.voice_config[speaker]['config'].style, # Correctly access style
                "use_speaker_boost": self.voice_config[speaker]['config'].use_speaker_boost, # Correctly access use_speaker_boost
            }

        )

        #convert generator to byte
        audio_byte= b''.join(chunk for chunk in audio_generator)

        filename = f"{self.output_dir}/{index:03d}_{speaker}.{self.audio_config.format}" # Correct f-string formatting
        with open(filename, 'wb') as out:
          out.write(audio_byte)

        # Basic audio Normalization
        if hasattr(self.audio_config, 'normalize') and self.audio_config.normalize: # Check if normalize exists and is True
          audio=AudioSegment.from_file(filename)
          normalized=audio.normalize() #normalize the audio seg
          normalized=normalized * 4  #slight boost

          # use context manager to ensure file closure
          with normalized.export(
              filename,
              format=self.audio_config.format,
              bitrate=self.audio_config.bitrate,
              parameters = ["-ar",str (self.audio_config.sample_rate)]
            ) as f:
              f.close()


        audio_files.append(filename)
        print(f"audio content written to file {filename}")

      except Exception as e:

        print(f"Error processing segment  {index} {str(e)}")
        continue

    return sorted(audio_files) # Return the list of generated audio files

In [ ]:
class PodcasMixer(BaseTool):
  "Enhanced audio mixing tool for podcast production"""

  name: str = "PodcastMixer"
  description: str = 'Mixes multiple audio files with effects into final podcast'

  audio_config : AudioConfig = Field(default_factory=AudioConfig)
  output_dir: str = Field(default="output/audio-files")


  def _run(
      self,
      audio_files : List[str],
      crossfade: int = 50 ,
  ) -> str:
      if not audio_files:
        raise ValueError("No audio files provided to mix")


      try:
        #create output directory if not exist

        os.makedirs(self.output_dir, exist_ok=True)

        mixed = AudioSegment.from_file(audio_files[0])
        for file in audio_files[1:]:
          next_segment=AudioSegment.from_file(file)

          # Add silence and use crossfade
          silence=AudioSegment.silent(duration=200)
          next_segment=silence + next_segment

          mixed=mixed.append(next_segment, crossfade=crossfade)


        # simplified output path handling
        output_file=os.path.join(self.output_dir, 'podcast_final.mp3')

        mixed.export(
            output_file,
            format='mp3',
            parameters={
                "-q-a","0", #Highest quality
                "-ar", "48000" #Professional sample rate
            }
        )

        print(f"successfully mixed podcast to {output_file}")
        return output_file

      except Exception as e:
       print(f"Error mixing podcast {str(e)}")
       return ""

OUTPUT DIRECTORY SETUP

In [ ]:
# setup for saving the directories

def setup_directories():
  """setting up organizised directory in a structured manner"""

  timestamp= datetime.now().strftime("%d/%m/%Y, %H:%M:%S")

  dirs = {
      'BASE':f"output/{timestamp}",
      'SEGMENT':f"ouput/{timestamp}/segment",
      'FINAL':f"output/{timestamp}/podcast",
      'DATA':f"output/{timestamp}/data"
  }

  for directory in dirs.values():
    os.makedirs(directory, exist_ok=True)

  return dirs

In [ ]:
# file uploading pipeline

import shutil
from google.colab import files

# creating the' knowledge' folder if not exist
if not os.path.exists("knowledge"):
  os.makedirs("knowledge")

# file uplaoding
uploaded=files.upload()
pdf_filename=list(uploaded.keys())[0]

# moving the uploaded file to knpwledge folder
shutil.move(pdf_filename, os.path.join("knowledge"), pdf_filename)

Saving 2507.23017v1.pdf to 2507.23017v1.pdf


'knowledge/2507.23017v1.pdf'

TASKS AND AGENT SETUP

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.knowledge.source.pdf_knowledge_source import PDFKnowledgeSource #pdf reader
from crewai_tools import SerperDevTool
from pydantic import BaseModel, Field
from typing import List
from datetime import datetime

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_config.py:323: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:151: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# passing the filename to pdfknowledgesource
research_paper=PDFKnowledgeSource(file_paths=pdf_filename)

PYDANTIC MODEL DEFINITION & STRUCTURE

In [ ]:
# Defining the pydandic model

class PaperSummary(BaseModel):
  """summary of the reseach paper"""

  title: str = Field(..., description="Title of the research papers")
  main_findings: List[str] = Field(..., description="key findings of the list of strings")
  methodology: str = Field(..., description="Research methods as a single text block")
  key_implication: str = Field(..., description="Key implications of list of strings")
  comparison: str = Field(..., description="Comparison of the research as a single text block")
  limitation: str = Field(..., description="Limitations of the list of strings")
  future_work: str = Field(..., description="Future research direction as a list")
  summary_date: datetime = Field(description="Timestamp of summary creation")


class Dialogueline(BaseModel):
  """dialogue lines for the paper summary"""

  speaker: str = Field(..., description="Name of Speaker(Lisa or Mark)")
  text: str = Field(..., description="The actual dialogue line")


class PodcastScript(BaseModel):
  """Podcast scripts with Dialogue lones"""

  dialogue: List[Dialogueline] = Field(..., description="Ordered list of dialogue lines")



class AudioGeneration(BaseModel):
  """Audio generation result with metadata"""

  segment_files: List[str] = Field(..., description="Lists of generated audio segments")
  final_podcast: List[str] = Field(..., description="Path to final mixed podcast audio")

LLM AGENTS CONFIGURATION

In [ ]:
# llms setup

summary_llm=LLM(
    model="openai/gpt-4o-mini"
)

script_llm=LLM(
    model="openai/o1"
)

script_enhancer_llm=LLM(
    model="openai/gpt-5-mini"
)

audio_generator_llm=LLM(
    model="openai/o4-mini"
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:151: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


SETUP FOR ELEVENLAB PODCAST HOST

In [ ]:
# configuring and creating tool

dirs=setup_directories()

audio_generator=PodcastAudioGeneration(
    output_dir=dirs['SEGMENT']
)

# Lisa : Expert
audio_generator.add_voice(
    "Lisa",
    voice_id=os.environ['Lisa_M_Voice_ID'],
    config=voiceConfig(
        stability=0.36,  #variation for natural enthusiast
        similarity_boost=0.75, #maintain voice consistency
        style=0.65,  #Goood expressiveness without being on over top of voice
        use_speaker_boost=True
    )
)


# Mark : Engaged and curious
audio_generator.add_voice(
    "Mark",
    voice_id=os.environ['Mark_Voice_ID'],
    config=voiceConfig(
        stability=0.4,  #slightly lower for more natural variation
        similarity_boost=0.85, #Higher to maintain consistence voice character
        style=0.6,  #Balance expression
        use_speaker_boost=True
    )
)


podcast_mixer=PodcasMixer(output_dir=dirs['FINAL'])
search_tool=SerperDevTool()

In [ ]:
# from crewai import Agent, Crew, Task, Process

AGENTS CONFIGURATION

In [ ]:
# setting up the agent that does the work

researcher = Agent(
    role="Research Analyst",
    goal="create comprehensive,intuitive yet accessible research paper summaries",
    backstory="""you're lead phD researcher with a talent for breaking down complex terms in academic paper into concise
    clear and understandable summaries.you thrive in identying KPIs,key findings and their real-world applications""",

    verbose=True,
    llm=summary_llm
)


research_support= Agent(
    role="Research Support Analyst",
    goal="Find current context and supporting documents relevant to the paper topics ",
    backstory="""you're a versatile research asistance who excels at finding supplementary and complementary
    information across various academic fields. You have a talent for connecting research to real-world applications
    current events and practical examples regardless of the field.You know how to find credible sources and relevant discussion
    across various domains""",
    verbose=True,
    tool=[search_tool],
    llm=script_llm
)


script_writer= Agent(
    role="Podcast Script Writer",
    goal="Create engaging and educational podcast scripts about technical topics",
    backstory="""you're a skilled and productive podcast writer that thrives in making technical
    contents engaging and accessible.you create natural and seemless dialogues between two hosts:
    Lisa (a knowledge expert who explains concepts and context clearly) and Mark (an informed co-host who asks thoughtful questions
    and helps guide the discussion)""",

    verbose=True,
    llm=script_llm


)

scripts_enhancer= Agent(
    role="Podcast Script Enhancer",
    goal="Enhances podcast scripts to be more engaging while maintaining educational value",
    backstory="""You're a veteran podcast producer who specialises in making technical contents
    both entertaining and informative.You excel at adding natural humor, relatable apologies,expressions,
    and engaging banters while ensuring the core technical content remains accurate and valuable
    you've worked on shows like Nvidia chaiman and google chairman researchs,Hardcore history,Lex's Fridman podcast
    and the joe ragan experience, bringing their signature blends of experience, humor,education,creativity...""",

    verbose=True,
    llm=script_enhancer_llm
)


audio_generator= Agent(
    role="Audio Podcast Generator",
    goal="Generate high quality podcast audio with natural sounding toned voice for each podcast segment",
    backstory="""You're an expert in audio generation and processing.you understand how to generate natural sounding
    voice and produce natual podcast audio.you consider all the natural attribute of speech like
    tone,pacing ,audio quality,pitch in production """,

    verbose=True,
    llm=audio_generator_llm
)



TASK OF THE AGENTS

In [ ]:
# clearifying tasking the agents with specific prompts

summary_task = Task(
    description=""" Hey there, researcher! your mision is to dive into the research paper
    provided in {paper} and uncover its core insights.
    As you create the summary pls:

      -Highlight the big ideas: what are main findings and conclusion?.
      -Explain the methods: Breakdown the study's methodology in everyday language.
      -Discuss the impacts: what are the key implications for the field?
      -Note of caveat: Note many limitations and uncertainties.
      Forward looking: offer some thoughts of future research areas.


    Keep your tone engaging and friendly so that an educated general audience can comprehend
    so they can easily follow along by staying on points of technicalities.""",

    expected_output="A clear, well structured summary that covers all the technical/critcal aspects of the paper",
    agent=researcher,
    output_pydantic=PaperSummary,
    output_file='output/metadata/paper_summary.json'
    )


supporting_research_task = Task(
    description="""Alright, now that we have the paper summary,lets add some real world flavor.
    your task is to gather credible and recent supporting materials that enriched the topic.
    Here's how to make it happen:

     -Spot the themes: Identify the main ideas from the papers and see how they connect with each other.
     -Current development: find news,publications,questions, answers,advancement or case study form the
     last couples of year and bring the idea to life.
     -Explore diverse views:Look for expert opinion,debates,conferences and alternative perspective.
     -Real world examples: Gather insights from industrial repos,reports,white papers or professional platforms.

    Your goal is to build collection of supporting insights that helps listeners/audiences understands how/what
    the research plays out in real life. Make sure your sources are recent, reliable, and add extra context.
    """,

   expected_output="A curated collection of supporting materials and real world examples that correlates",
   agent=research_support,
   context=[summary_task],
   output_file='output/metadata/supporting_research.json'
)


podscast_task = Task(
    description="""Using the paper summary and supporting research, craft a world-class podcast conversation"

    Adopt a conversational:
    -use natural expressions and filters(e.g , oh, yeah,like,wow,wait,really?,laugh,hold-on etc) to keep it natural
    -include friendly interruptions,confirmations,affirmations and asides just like when two people chatting

   clearly distinguish sources:
   -when referencing the paper, use statement like 'according to the paper...' or' the study of the paper...'
   -when referencing supporting material, use statement like 'based on paper','i read somewhere that...'

   Embrace the host personalities:
   -Lisa:An Expert who expains technical details in a concise, approachable way sometimes playfully challenging Marks
   -Mark:An informed co-host who asks thoughtful questions and helps guide the discussion and brings in relatable real-world scenario.

   Keep your tone engaging and friendly so that an educated general audience can comprehend
   so they can easily follow along by staying on points of technicalities.

   Your script should clearly separate the paper summary from the broader insights while engaging the listeners with a genuine
   and relaxed conversation.""",

   expected_output="A clear,lively well structured,natural podcast scripts that seemlessly blend\
    all the technical/critcal aspects of the paper",
   agent=script_writer,
   context=[summary_task,supporting_research_task],
   output_pydantic=PodcastScript,
   output_file='output/metadata/podcast_script.json'
 )


enhanced_script_task = Task(
    description="""Now, take the initial podcast script and polish it further to make it feel genuine, off-the-cut
    conversation between Lisa and Mark.

    Your job is to:
    -Infuse natural reaction:sprinkle casual phrases like cool,nice,oh,christ,oh my God,oh wait!,that's interesting
    to capture spontaneous moments.
    -Smooth the flow:Ensure the dialogue transition smoothly between topics with natural pause and brief asides.
    -Keep it authentic:Maintain the distinct personalty of Lisa(friendly expert) and Mark(curious co-host), without
    adding any extra character.
    -Preserve accuracy:While making the dialogue fun and engaging always make sure all technical details is correct.
    Inject light humor: when it fits natural, add humorous or witty exchanges to keep the conversation lively.


    Always Remember:Do not change the host names and avoid explicit reaction markers like **laugh**.Let the dialogue
    implies the emotion naturally.

    your enhanced scripts should sound like you're eavesdroping on genuine, spontanous chats between two friends
    who are both passionate about the topics.
    """,

    expected_output="An enhanced podcast scripts that feels natural, engaging and authentic",
    context=[summary_task, podscast_task],
    output_pydantic=PodcastScript,
    agent=scripts_enhancer,
    output_file='output/metadata/enhanced_podcast_script.json',
    human_input=False
    ),


audio_task= Task(
    description="""Generate high quality audio for the podcast script and create the final podcast

    The scripts will be provided in the context as a list of dialogue entries, each with:
    -speaker: Lisa or Mark
    -text: The actual dialogue line

    Your task is to:
    -Generate natural sounding audio for line of dialogue using appropriate voices.
    -Apply audio processing for professional quality:
        1.Normalise audio levels
        2.Add subtle fade effect between every segments.
        3.Apply appropriate pacing and pause

   -Mix all segment into a cohesive final podcast

   Voice assignment:
   -For Lisa lines: use configured Lsa's voice
  -For Mark lines: use configured Mark'

  Quality Guidelines:
  -Ensure consistence audio levels across all segments.
  -Maintain natural pacing and flow.
  -Create smooth transitions between speakers.
  -Verify audio clarity and quality.
  """,

  expected_output="A professional high-quality audio podcasts files with natural sounding voice",
  context=[enhanced_script_task],
  output_pydantic=AudioGeneration,
  agent=audio_generator,
  output_file='output/metadata/audio_generation_meta.json'
)

AttributeError: 'tuple' object has no attribute 'get'